# 03. Exploración de la temperatura de Open-Meteo

Consulta del reanálisis ERA5 a través de la API de Open-Meteo y su agregación mensual (tarea 4 del backlog, HU-01). La temperatura se pide en las coordenadas de la estación Trelew Aero del SMN. La lógica definitiva se encuentra en `src/carga_openmeteo.py`.

**Atención:** ejecutar esta notebook vuelve a consultar la API. El archivo original solo se guarda si todavía no existe, para evitar sobrescribirlo.

In [1]:
import sys
sys.path.append("../src")

In [2]:
import pandas as pd
import requests
from pathlib import Path
from carga_openmeteo import obtener_era5
from carga_openmeteo import guardar_consulta
from carga_openmeteo import calculos_mensuales
from carga_cammesa import csv_tabla

## 1. Consulta a la API

Se piden la temperatura media, máxima y mínima diaria de enero 2019 a septiembre 2026, con el modelo ERA5 y la zona horaria de Argentina.

In [3]:
diccionario = {
    "latitude" : -43.20894,
    "longitude" : -65.28209,
    "start_date" : "2019-01-01",
    "end_date" : "2026-09-30",
    "timezone" : "America/Argentina/Buenos_Aires",
    "daily" : ['temperature_2m_mean','temperature_2m_max','temperature_2m_min'],
    "models" : "era5"
}

url_base = "https://archive-api.open-meteo.com/v1/archive"

consulta = requests.get(url_base,params=diccionario)
consulta

<Response [200]>

In [4]:
openmeteo = consulta.json()
openmeteo.keys()

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])

La respuesta trae la ubicación de la celda ERA5 que usó la API y los datos diarios.

In [5]:
print('Latitud: ',openmeteo["latitude"])
print('Longitud: ',openmeteo["longitude"])
print('Zona horaria: ',openmeteo["timezone"])
print('Desfase respecto a UTC: ',openmeteo["utc_offset_seconds"])
print('Daily',openmeteo['daily'].keys())

Latitud:  -43.25
Longitud:  -65.25
Zona horaria:  America/Argentina/Buenos_Aires
Desfase respecto a UTC:  -10800
Daily dict_keys(['time', 'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min'])


## 2. Agregación mensual

Se arma una tabla diaria y se resume por mes con el promedio de las medias, el máximo de las máximas y el mínimo de las mínimas. Se usan los extremos porque la variable a pronosticar es el pico de demanda del mes, que sale del día con mayor demanda.

In [6]:
daily = pd.DataFrame(openmeteo['daily'])
daily['time'] = pd.to_datetime(daily['time'])
daily.head()

,time,temperature_2m_mean,temperature_2m_max,temperature_2m_min
0,2019-01-01,19.0,24.6,14.2
1,2019-01-02,15.9,20.5,11.7
2,2019-01-03,17.3,23.7,11.7
3,2019-01-04,22.4,30.7,11.7
4,2019-01-05,20.4,23.5,17.8


In [7]:
daily.info()

<class 'pandas.DataFrame'>
RangeIndex: 2830 entries, 0 to 2829
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   time                 2830 non-null   datetime64[us]
 1   temperature_2m_mean  2830 non-null   float64       
 2   temperature_2m_max   2830 non-null   float64       
 3   temperature_2m_min   2830 non-null   float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 88.6 KB


In [8]:
calculos = {"temperature_2m_mean" : "mean",
"temperature_2m_max" : "max",
"temperature_2m_min" : "min"}
temperaturas = daily.set_index("time").resample("MS").agg(calculos)
temperaturas.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 93 entries, 2019-01-01 to 2026-09-01
Freq: MS
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   temperature_2m_mean  93 non-null     float64
 1   temperature_2m_max   93 non-null     float64
 2   temperature_2m_min   93 non-null     float64
dtypes: float64(3)
memory usage: 2.9 KB


## 3. Prueba de las funciones del módulo

Se repite el proceso con las funciones de `src/carga_openmeteo.py`. La función `obtener_era5` consulta la API, `guardar_consulta` guarda la respuesta original y `calculos_mensuales` hace la agregación. El resultado debe contener los mismos 93 meses.

In [9]:
openmeteo = obtener_era5("2019-01-01","2026-09-30")

In [10]:
len(openmeteo["daily"]["time"])

2830

In [11]:
ruta = Path("../data/raw/openmeteo/OPENMETEO_ERA5_temp_2019-01-01_2026-09-30_2026-10-22.json")
if not ruta.exists():
    guardar_consulta(openmeteo, ruta)

In [12]:
df = calculos_mensuales(openmeteo)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   MES                  93 non-null     datetime64[us]
 1   TEMPERATURA_MEDIA_C  93 non-null     float64       
 2   TEMPERATURA_MAX_C    93 non-null     float64       
 3   TEMPERATURA_MIN_C    93 non-null     float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 3.0 KB


In [13]:
ruta_salida = Path("../data/processed/OPENMETEO_TRELEW_TempMensual.csv")
csv_tabla(df,ruta_salida)

## Conclusión

- La API devolvió 2.830 días, que se agregan en 93 meses, los mismos de la serie de CAMMESA.
- Las coordenadas pedidas son las de Trelew Aero (-43,20894, -65,28209), pero la API respondió con la celda ERA5 centrada en -43,25 y -65,25. La temperatura corresponde a esa celda, no a la estación.
- La temperatura es un reanálisis: una estimación que combina observaciones y modelos numéricos, no una medición directa de la estación.